<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-04/blob/main/hw_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 4 — DML, DDL and Complex SQL Expressions

## 1. Setup and Data Loading

In [2]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 13.5 MB/s eta 0:00:00


In [3]:
!pip install -q psycopg2-binary sqlalchemy pandas kagglehub

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 21.2 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [4]:
# Dataset import
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [5]:
import os

sorted(os.listdir(dataset_dir))

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [6]:
files = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_order_reviews_dataset.csv',
]

for file_name in files:
    df_tmp = pd.read_csv(os.path.join(dataset_dir, file_name))
    print(f"\n{file_name}")
    print("shape:", df_tmp.shape)
    print("columns:", df_tmp.columns.tolist())


olist_customers_dataset.csv
shape: (99441, 5)
columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_orders_dataset.csv
shape: (99441, 8)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_order_items_dataset.csv
shape: (112650, 7)
columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_products_dataset.csv
shape: (32951, 9)
columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

olist_sellers_dataset.csv
shape: (3095, 4)
columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

olist_order_reviews_dataset.csv
shape: (99224, 7)
columns

In [7]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000
    )

    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


### 1.2 Create Typed Tables with Constraints

In [8]:
with engine.begin() as conn:
    conn.execute(text("""

      DROP TABLE IF EXISTS customer_segments    CASCADE;
      DROP TABLE IF EXISTS seller_score         CASCADE;
      DROP TABLE IF EXISTS seller_alerts        CASCADE;
      DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
      DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
      DROP TABLE IF EXISTS olist_order_items    CASCADE;
      DROP TABLE IF EXISTS olist_orders         CASCADE;
      DROP TABLE IF EXISTS olist_customers      CASCADE;
      DROP TABLE IF EXISTS olist_products       CASCADE;
      DROP TABLE IF EXISTS olist_sellers        CASCADE;

      CREATE TABLE olist_customers (
          customer_id              TEXT PRIMARY KEY,
          customer_unique_id       TEXT NOT NULL,
          customer_zip_code_prefix INTEGER,
          customer_city            TEXT,
          customer_state           CHAR(2) NOT NULL
      );

      CREATE INDEX idx_olist_customers_unique_id
          ON olist_customers (customer_unique_id);

      CREATE TABLE olist_orders (
          order_id                      TEXT PRIMARY KEY,
          customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
          order_status                  TEXT NOT NULL,
          order_purchase_timestamp      TIMESTAMP NOT NULL,
          order_approved_at             TIMESTAMP,
          order_delivered_carrier_date  TIMESTAMP,
          order_delivered_customer_date TIMESTAMP,
          order_estimated_delivery_date TIMESTAMP,
          CHECK (order_status IN (
              'created', 'approved', 'invoiced', 'processing',
              'shipped', 'delivered', 'unavailable', 'canceled'
          ))
      );

      CREATE TABLE olist_products (
          product_id                 TEXT PRIMARY KEY,
          product_category_name      TEXT,
          product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
          product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
          product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
          product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
          product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
          product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
          product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
      );

      CREATE TABLE olist_sellers (
          seller_id              TEXT PRIMARY KEY,
          seller_zip_code_prefix INTEGER,
          seller_city            TEXT,
          seller_state           CHAR(2) NOT NULL
      );

      CREATE TABLE olist_order_items (
          order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
          order_item_id       INTEGER NOT NULL,
          product_id          TEXT REFERENCES olist_products(product_id),
          seller_id           TEXT REFERENCES olist_sellers(seller_id),
          shipping_limit_date TIMESTAMP,
          price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
          freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
          PRIMARY KEY (order_id, order_item_id)
      );

      CREATE INDEX idx_olist_order_items_seller_order
          ON olist_order_items (seller_id, order_id);

      CREATE TABLE olist_order_reviews (
          review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
          review_id                TEXT,
          order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
          review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
          review_comment_title     TEXT,
          review_comment_message   TEXT,
          review_creation_date     TIMESTAMP,
          review_answer_timestamp  TIMESTAMP
      );

      CREATE INDEX idx_olist_order_reviews_order_id
          ON olist_order_reviews (order_id);

    """))
print("Successfully created typed tables.")

Successfully created typed tables.


### 1.3 Load Data from Raw to Typed Tables

In [9]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO olist_customers
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix::INTEGER,
            customer_city,
            customer_state::CHAR(2)
        FROM olist_customers_raw;

        INSERT INTO olist_orders
        SELECT
            order_id,
            customer_id,
            order_status,
            order_purchase_timestamp::TIMESTAMP,
            order_approved_at::TIMESTAMP,
            order_delivered_carrier_date::TIMESTAMP,
            order_delivered_customer_date::TIMESTAMP,
            order_estimated_delivery_date::TIMESTAMP
        FROM olist_orders_raw;

        INSERT INTO olist_products
        SELECT
            product_id,
            product_category_name,
            product_name_lenght::INTEGER,
            product_description_lenght::INTEGER,
            product_photos_qty::INTEGER,
            product_weight_g::INTEGER,
            product_length_cm::INTEGER,
            product_height_cm::INTEGER,
            product_width_cm::INTEGER
        FROM olist_products_raw;

        INSERT INTO olist_sellers
        SELECT
            seller_id,
            seller_zip_code_prefix::INTEGER,
            seller_city,
            seller_state::CHAR(2)
        FROM olist_sellers_raw;

        INSERT INTO olist_order_items
        SELECT
            order_id,
            order_item_id::INTEGER,
            product_id,
            seller_id,
            shipping_limit_date::TIMESTAMP,
            price::NUMERIC(12, 2),
            freight_value::NUMERIC(12, 2)
        FROM olist_order_items_raw;

        INSERT INTO olist_order_reviews (
            review_id,
            order_id,
            review_score,
            review_comment_title,
            review_comment_message,
            review_creation_date,
            review_answer_timestamp
        )
        SELECT
            review_id,
            order_id,
            review_score::SMALLINT,
            review_comment_title,
            review_comment_message,
            review_creation_date::TIMESTAMP,
            review_answer_timestamp::TIMESTAMP
        FROM olist_order_reviews_raw;
    """))

print("Raw data successfully loaded into typed tables.")

Raw data successfully loaded into typed tables.


### 1.4 Validate Typed Table Row Counts

In [10]:
pd.read_sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers

    UNION ALL

    SELECT 'orders', COUNT(*)
    FROM olist_orders

    UNION ALL

    SELECT 'order_items', COUNT(*)
    FROM olist_order_items

    UNION ALL

    SELECT 'products', COUNT(*)
    FROM olist_products

    UNION ALL

    SELECT 'sellers', COUNT(*)
    FROM olist_sellers

    UNION ALL

    SELECT 'reviews', COUNT(*)
    FROM olist_order_reviews

    ORDER BY table_name;
""", engine)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


All six typed tables contain the same number of rows as their corresponding raw tables. No records were lost during type conversion and loading.

## 2. DML queries with PostgreSQL features

### 2.1 INSERT ... RETURNING

In [11]:
with engine.begin() as conn:
    # Create the alerts table
    conn.execute(text("""
        DROP TABLE IF EXISTS seller_alerts CASCADE;

        CREATE TABLE seller_alerts (
            alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
            alert_type TEXT NOT NULL
                CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
            severity   SMALLINT NOT NULL
                CHECK (severity BETWEEN 1 AND 5),
            details    TEXT,
            created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );
    """))

    # Find the seller with the most late deliveries and create an alert
    result = conn.execute(text("""
        WITH seller_orders AS (
            SELECT DISTINCT
                oi.seller_id,
                oi.order_id
            FROM olist_order_items AS oi
        ),
        late_sellers AS (
            SELECT
                so.seller_id,
                COUNT(DISTINCT o.order_id) AS late_orders
            FROM seller_orders AS so
            JOIN olist_orders AS o
                ON o.order_id = so.order_id
            WHERE o.order_delivered_customer_date IS NOT NULL
              AND o.order_estimated_delivery_date IS NOT NULL
              AND o.order_delivered_customer_date::DATE
                    > o.order_estimated_delivery_date::DATE
            GROUP BY so.seller_id
            ORDER BY late_orders DESC, so.seller_id
            LIMIT 1
        )
        INSERT INTO seller_alerts (
            seller_id,
            alert_type,
            severity,
            details
        )
        SELECT
            seller_id,
            'late_delivery',
            4,
            'Seller has ' || late_orders ||
            ' late delivered orders in the loaded dataset'
        FROM late_sellers
        RETURNING
            alert_id,
            seller_id,
            alert_type,
            severity,
            details,
            created_at;
    """))

    rows = result.fetchall()
    columns = result.keys()

pd.DataFrame(rows, columns=columns)

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-09-29 10:27:41.847445+00:00


The query created a late_delivery alert for the seller with the highest number of late delivered orders. The seller had 172 late deliveries. RETURNING exposed the inserted row immediately, including the generated alert_id and timestamp, which makes the operation easy to audit in the notebook.

### 2.2 INSERT ... ON CONFLICT DO UPDATE

In [34]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS seller_score CASCADE;

        CREATE TABLE seller_score (
            seller_id   TEXT PRIMARY KEY
                REFERENCES olist_sellers(seller_id),
            avg_rating  NUMERIC(3, 2),
            n_reviews   INTEGER NOT NULL DEFAULT 0
                CHECK (n_reviews >= 0),
            tier        TEXT
                CHECK (tier IN ('gold', 'silver', 'bronze')),
            updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );
    """))

    result = conn.execute(text("""
        WITH seller_orders AS (
            SELECT DISTINCT
                oi.seller_id,
                oi.order_id
            FROM olist_order_items AS oi
        ),

        review_per_order AS (
            SELECT
                rv.order_id,
                AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
            FROM olist_order_reviews AS rv
            WHERE rv.review_score IS NOT NULL
            GROUP BY rv.order_id
        ),

        seller_rating AS (
            SELECT
                so.seller_id,
                AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
                COUNT(rpo.order_id) AS n_reviews
            FROM seller_orders AS so
            LEFT JOIN review_per_order AS rpo
                ON rpo.order_id = so.order_id
            GROUP BY so.seller_id
        )

        INSERT INTO seller_score (
            seller_id,
            avg_rating,
            n_reviews,
            tier
        )
        SELECT
            seller_id,
            avg_rating,
            n_reviews,
            CASE
                WHEN avg_rating IS NULL THEN NULL
                WHEN avg_rating >= 4.5 THEN 'gold'
                WHEN avg_rating >= 3.5 THEN 'silver'
                ELSE 'bronze'
            END AS tier
        FROM seller_rating

        ON CONFLICT (seller_id) DO UPDATE
        SET
            avg_rating = EXCLUDED.avg_rating,
            n_reviews  = EXCLUDED.n_reviews,
            tier       = EXCLUDED.tier,
            updated_at = NOW()

        RETURNING
            seller_id,
            avg_rating,
            n_reviews,
            tier,
            updated_at;
    """))

    rows = result.fetchall()
    columns = result.keys()

seller_score_df = pd.DataFrame(rows, columns=columns)

seller_score_df.head(10)

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-09-29 11:17:02.379616+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-09-29 11:17:02.379616+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-09-29 11:17:02.379616+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-09-29 11:17:02.379616+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-09-29 11:17:02.379616+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-09-29 11:17:02.379616+00:00
6,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-09-29 11:17:02.379616+00:00
7,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-09-29 11:17:02.379616+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-09-29 11:17:02.379616+00:00
9,bf0d50a6410d487dc97d2baac0a8c0be,2.00,1,bronze,2026-09-29 11:17:02.379616+00:00


In [23]:
# Disrtibution by tier
with engine.begin() as conn:
    result = conn.execute(text("""
      SELECT
          COALESCE(tier, 'no_rating') AS tier_group,
          COUNT(*) AS seller_count
      FROM seller_score
      GROUP BY tier
      ORDER BY seller_count DESC;
    """))
    rows = result.fetchall()
    columns = result.keys()

pd.DataFrame(rows, columns=columns)


,tier_group,seller_count
0,silver,1568
1,gold,983
2,bronze,539
3,no_rating,5


Most sellers fall into the silver tier, followed by gold and bronze. Only 5 sellers have no rating, which indicates that review coverage is high across the seller population.

### 2.3 UPDATE ... FROM